In [1]:
import pickle
import numpy as np
from sklearn.model_selection import LeaveOneGroupOut
from xgboost import XGBRegressor

from ppghr.features import FEATURE_NAMES
from ppghr.io import ACTIVITIES, PROJECT_ROOT

with open(PROJECT_ROOT / "data" / "processed" / "features.pkl", "rb") as f:
    d = pickle.load(f)
X, y, act, groups = d["X"], d["y"], d["act"], d["groups"]

p = np.load(PROJECT_ROOT / "results" / "metrics" / "predictions.npz")
pred_full, baseline = p["pred_full"], p["baseline"]

print(X.shape, np.abs(pred_full - y).mean().round(3))

(64697, 21) 8.337


In [2]:
names = list(FEATURE_NAMES)
i_base = names.index("baseline_est_bpm")
i_energy = names.index("acc_total_energy")

def _fit(Xa, ya):
    m = XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.8,
                     objective="reg:absoluteerror", n_jobs=-1, random_state=0)
    m.fit(Xa, ya)
    return m

def loso_hybrid(X, y, groups, n_inner=3):
    pred = np.zeros(len(y))
    thresholds = []
    for tr, te in LeaveOneGroupOut().split(X, y, groups):
        tr_sids = np.unique(groups[tr])
        val_sids = tr_sids[-n_inner:]
        val_m = np.isin(groups, val_sids)
        inner_tr = tr[~np.isin(groups[tr], val_sids)]

        m_in = _fit(X[inner_tr], y[inner_tr])
        p_val = m_in.predict(X[val_m])
        e_val = X[val_m, i_energy]

        best_t, best_mae = np.inf, np.inf
        for q in range(0, 101, 5):
            t = np.percentile(X[inner_tr, i_energy], q)
            blend = np.where(e_val < t, X[val_m, i_base], p_val)
            mae = np.abs(blend - y[val_m]).mean()
            if mae < best_mae:
                best_mae, best_t = mae, t
        thresholds.append(best_t)

        m = _fit(X[tr], y[tr])
        p_te = m.predict(X[te])
        pred[te] = np.where(X[te, i_energy] < best_t, X[te, i_base], p_te)
    return pred, np.array(thresholds)

pred_hyb, ths = loso_hybrid(X, y, groups)
print(f"full model  {np.abs(pred_full - y).mean():.3f}")
print(f"hybrid      {np.abs(pred_hyb - y).mean():.3f}")
print(f"thresholds chosen per fold: {np.percentile(ths, [0,50,100]).round(3)}")

full model  8.337
hybrid      8.352
thresholds chosen per fold: [0. 0. 0.]


In [3]:
for a in ["sitting", "working", "walking", "stairs"]:
    m = act == {v: k for k, v in ACTIVITIES.items()}[a]
    print(f"{a:10s} base {np.abs(baseline[m]-y[m]).mean():6.2f}  "
          f"full {np.abs(pred_full[m]-y[m]).mean():6.2f}  "
          f"hybrid {np.abs(pred_hyb[m]-y[m]).mean():6.2f}")

sitting    base   2.96  full   4.09  hybrid   4.14
working    base   5.99  full   4.46  hybrid   4.51
walking    base  23.53  full  11.55  hybrid  11.55
stairs     base  31.81  full  17.88  hybrid  17.88
